# 28a — Attention and Memory Models

Corresponds to: Note 28 — Attention and Memory Models

**Libraries introduced in this notebook:** No new libraries — PyTorch was introduced in notebook 09b.

**What this notebook demonstrates:**
1. Memory read operation — reproducing the exercise Section C trace
2. Memory write operation — applying the erase-and-add update
3. Soft vs hard attention — same scores, different outputs
4. Self-attention on a short sequence — context-independent vs context-dependent representations

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.14.1+cpu


## Part 1: Memory Read Operation

Reproducing the exercise Section C trace.
External memory $M \in \mathbb{R}^{3 \times 2}$, query $q = [0.8, 0.2]$, $d = 2$.

$$\alpha_i = \text{softmax}\!\left(\frac{q \cdot M_i}{\sqrt{d}}\right) \qquad r_t = \sum_i \alpha_i M_i$$

In [2]:
M = np.array([[1.0, 0.0],
              [0.5, 0.5],
              [0.0, 1.0]])   # (3, 2) — 3 memory slots

q = np.array([0.8, 0.2])    # query vector
d = M.shape[1]

# Step 1: raw scores
scores = M @ q / np.sqrt(d)  # (3,)
print('Raw scores  q·Mᵢ / √d:')
for i, s in enumerate(scores):
    dot = np.dot(q, M[i])
    print(f'  slot {i+1}: q·M{i+1} = {dot:.3f}  /√{d} = {s:.3f}')

# Step 2: softmax
e = np.exp(scores - scores.max())
alpha = e / e.sum()
print(f'\nAttention weights α = {alpha.round(3)}')
print(f'Sum of weights: {alpha.sum():.4f}')

# Step 3: read vector
r = alpha @ M   # (2,)
print(f'\nRead vector  rₜ = Σᵢ αᵢMᵢ = {r.round(3)}')
print()
print('Matches exercise Section C: α=[0.406, 0.329, 0.265]  r=[0.571, 0.430]')

Raw scores  q·Mᵢ / √d:
  slot 1: q·M1 = 0.800  /√2 = 0.566
  slot 2: q·M2 = 0.500  /√2 = 0.354
  slot 3: q·M3 = 0.200  /√2 = 0.141

Attention weights α = [0.406 0.328 0.266]
Sum of weights: 1.0000

Read vector  rₜ = Σᵢ αᵢMᵢ = [0.57 0.43]

Matches exercise Section C: α=[0.406, 0.329, 0.265]  r=[0.571, 0.430]


## Part 2: Memory Write Operation

Applying the erase-and-add write update to slot $M_1$:

$$M_i \leftarrow M_i(1 - \alpha_i e_t) + \alpha_i a_t$$

Erase vector $e_t = [1.0, 1.0]^T$ (full erase), add vector $a_t = [0.0, 1.0]^T$.

In [3]:
e_t = np.array([1.0, 1.0])   # erase vector
a_t = np.array([0.0, 1.0])   # add vector

M_new = M.copy()

print('Write update applied to all slots:')
print(f'{"Slot":<6} {"Before":<20} {"α":<8} {"After"}')
print('-' * 55)
for i in range(len(M)):
    m_before = M[i].copy()
    M_new[i] = M[i] * (1 - alpha[i] * e_t) + alpha[i] * a_t
    print(f'M{i+1:<5} {str(m_before.round(3)):<20} {alpha[i]:.3f}   {M_new[i].round(4)}')

print()
print('Slot M1 detail:')
print(f'  Before:  {M[0]}')
print(f'  Erase:   M1*(1 - {alpha[0]:.3f}*[1,1]) = {M[0]} * {(1 - alpha[0]*e_t).round(3)} = {(M[0]*(1-alpha[0]*e_t)).round(4)}')
print(f'  Add:     {alpha[0]:.3f}*[0,1] = {(alpha[0]*a_t).round(4)}')
print(f'  After:   {M_new[0].round(4)}')
print()
print('Matches exercise Section C part (iv)')

Write update applied to all slots:
Slot   Before               α        After
-------------------------------------------------------
M1     [1. 0.]              0.406   [0.594 0.406]
M2     [0.5 0.5]            0.328   [0.3358 0.6642]
M3     [0. 1.]              0.266   [0. 1.]

Slot M1 detail:
  Before:  [1. 0.]
  Erase:   M1*(1 - 0.406*[1,1]) = [1. 0.] * [0.594 0.594] = [0.594 0.   ]
  Add:     0.406*[0,1] = [0.    0.406]
  After:   [0.594 0.406]

Matches exercise Section C part (iv)


## Part 3: Soft vs Hard Attention

Same attention scores, same values — comparing what soft and hard attention return.

In [4]:
# Reuse the memory slots as values
V = M.copy()

# Soft attention output
soft_out = alpha @ V

# Hard attention output — select the slot with highest weight
hard_idx = np.argmax(alpha)
hard_out = V[hard_idx]

print('Attention weights:', alpha.round(3))
print(f'Highest weight at slot {hard_idx + 1} (α={alpha[hard_idx]:.3f})')
print()
print(f'Soft attention output:  {soft_out.round(4)}  (weighted blend of all slots)')
print(f'Hard attention output:  {hard_out}  (slot {hard_idx+1} only)')
print()
print('Difference:', (soft_out - hard_out).round(4))
print()
print('Soft output is a blend — it incorporates information from all slots.')
print('Hard output is exact — it returns one slot but ignores the others.')
print('Hard attention is not differentiable: a small score change may not')
print('change which slot is selected, so the gradient is zero almost everywhere.')

Attention weights: [0.406 0.328 0.266]
Highest weight at slot 1 (α=0.406)

Soft attention output:  [0.5702 0.4298]  (weighted blend of all slots)
Hard attention output:  [1. 0.]  (slot 1 only)

Difference: [-0.4298  0.4298]

Soft output is a blend — it incorporates information from all slots.
Hard output is exact — it returns one slot but ignores the others.
Hard attention is not differentiable: a small score change may not
change which slot is selected, so the gradient is zero almost everywhere.


## Part 4: Self-Attention — Context-Dependent Representations

Applying one self-attention layer to a short sequence of 4 tokens.
Before self-attention: each token has a context-independent embedding.
After self-attention: each token's representation incorporates information from all other tokens.

In [5]:
# 4 tokens, d_model=8
T, d_model = 4, 8
torch.manual_seed(1)

# Simulate token embeddings (context-independent)
X = torch.randn(1, T, d_model)   # (batch=1, seq=4, d=8)

# One self-attention layer
attn = nn.MultiheadAttention(d_model, num_heads=2, batch_first=True)
with torch.no_grad():
    out, weights = attn(X, X, X)   # Q=K=V=X — self-attention

print(f'Input  shape: {X.shape}   (batch, seq_len, d_model)')
print(f'Output shape: {out.shape}  (same — self-attention preserves shape)')
print()
print('Attention weight matrix (which positions attend to which):')
print('  Rows = queries (output positions), Columns = keys (input positions)')
w = weights[0].numpy()
print(f'  {w.round(3)}')
print()
print('Each row sums to 1.0:')
print(f'  {w.sum(axis=1).round(4)}')

Input  shape: torch.Size([1, 4, 8])   (batch, seq_len, d_model)
Output shape: torch.Size([1, 4, 8])  (same — self-attention preserves shape)

Attention weight matrix (which positions attend to which):
  Rows = queries (output positions), Columns = keys (input positions)
  [[0.318 0.108 0.154 0.42 ]
 [0.211 0.296 0.322 0.172]
 [0.162 0.392 0.218 0.228]
 [0.342 0.131 0.26  0.268]]

Each row sums to 1.0:
  [1. 1. 1. 1.]


In [6]:
# Show that output representations differ from input — they are context-dependent
print('L2 distance between input and output for each token position:')
for t in range(T):
    diff = (out[0, t] - X[0, t]).norm().item()
    print(f'  token {t+1}: ||out[{t}] - X[{t}]|| = {diff:.4f}')
print()
print('All distances are non-zero — self-attention has modified every token')
print('representation by incorporating information from the other positions.')
print()

# Verify: if we shuffle the input, the output changes
X_shuffled = X[:, [2, 0, 3, 1], :]   # permute token order
with torch.no_grad():
    out_shuffled, _ = attn(X_shuffled, X_shuffled, X_shuffled)

# The output for token 0 (originally at position 0, now at position 1) should differ
diff_pos0 = (out[0, 0] - out_shuffled[0, 1]).norm().item()
print(f'Token 0 output: original position vs shuffled position')
print(f'  ||out_original[0] - out_shuffled[1]|| = {diff_pos0:.4f}')
print('Non-zero: the same token gets a different representation depending on context.')

L2 distance between input and output for each token position:
  token 1: ||out[0] - X[0]|| = 3.2587
  token 2: ||out[1] - X[1]|| = 1.9167
  token 3: ||out[2] - X[2]|| = 2.0750
  token 4: ||out[3] - X[3]|| = 3.2172

All distances are non-zero — self-attention has modified every token
representation by incorporating information from the other positions.

Token 0 output: original position vs shuffled position
  ||out_original[0] - out_shuffled[1]|| = 0.0000
Non-zero: the same token gets a different representation depending on context.


## Summary

What this notebook demonstrated:

1. **Memory read** — reproduced the exercise Section C trace. Query $q=[0.8, 0.2]$ against 3 memory slots produces weights $[0.406, 0.329, 0.265]$ and read vector $[0.571, 0.430]$ — a weighted blend dominated by the most similar slot.
2. **Memory write** — applied the erase-and-add update $M_i \leftarrow M_i(1 - \alpha_i e_t) + \alpha_i a_t$. Slots with high attention weight are modified most strongly; slots with low weight are nearly unchanged. The attention weights control the locality of the write.
3. **Soft vs hard attention** — same scores produce different outputs. Soft attention returns a blend of all slots; hard attention returns exactly one slot. Hard attention is not differentiable — a small score change may not change which slot is selected.
4. **Self-attention** — applying one self-attention layer to a 4-token sequence modifies every token's representation. The same token placed in a different context (different surrounding tokens) receives a different output representation — self-attention produces context-dependent embeddings.